# Nightly research Q01: shared ECI library + reproduction check

**Question (QUEUE.md Q01):** Port the lab's `compute_mcp` / `compute_eci` from `code/current/Spring2026.ipynb` into `willbetts/nightly/lib/eci.py` with a sign fix (orient to diversity), add an HS4 ECI builder from `data/raw`, and check that it reproduces (a) the grouped ECI the notebook produces and (b) Maitree's HS4 vs grouped income correlations.

- **Date:** 2026-11-11 (nightly agent)
- **Data:** `data/cleaned/<year>clean.csv` and the identical copies in `data/archive/2024-fall/2019-2023 cleaned/` (md5-identical, checked before the run); `data/raw/<year>_mega.csv`; `maitreemody/outputs/country_year_metrics.csv`, `correlations.csv`, `worldbank_gdp_per_capita_2019_2023.csv`; report text from `reports/2026-spring/*.pdf` and `catalog/text/reports/2025-fall/Singapore/Singapore_2023.md`
- **Years:** 2019–2023 (2023 suspect, D1)
- **Run time:** about 5 seconds (printed in the last code cell)
- **New file this question creates:** `willbetts/nightly/lib/eci.py`

## TL;DR

- **The library reproduces the notebook exactly.** Running the notebook's own `df_to_matrix` → `compute_mcp` → `compute_mcc` → `compute_eci` (parsed out of `Spring2026.ipynb`) and `eci.notebook_eci` on the same data gives the same country sets (121/121/106/121/123 countries for 2019–2023) and a max absolute difference of at most 6.2e-15 (Section A).
- **The notebook's eigenvector is the right one, but its sign flips in 2021, 2022 and 2023.** Column 1 of `scipy.linalg.eig` happened to be the 2nd-largest eigenvalue in every year, so only the sign is wrong. Notebook ECI is perfectly anti-correlated (−1.0) with diversity-oriented ECI in 2021–2023 (Section B). The sign flip lines up with the "2020→2021 ECI drop" in KNOWN-ISSUES D3 (the shaded half of the figure). With the sign fixed, the Spring 2026 countries' paths are smooth across 2020→2021.
- **The library also reproduces Maitree's HS4 and grouped ECI to ≤1.5e-14** in all 5 years, and her HS4 Spearman correlations with log GDPpc exactly (0.7795, 0.8057, 0.7659, 0.7753, 0.7661). Grouped correlations differ in the 3rd decimal (e.g. 0.3730 vs 0.3718 in 2019) only because 4–9 countries per year have tied grouped ECIs and float noise breaks the ties. Rounding to 9 dp makes them identical (Section D).
- **⚠️ The published Spring 2026 report ECIs can't be reproduced from the data in the repo.** None of the 14 start/end values matches the same country in any of the 30 year × variant combinations we tried. Those reports say "2019 to 2024", but the repo has no 2024 data. The Fall 2025 Singapore report's ECI percentage changes have the right direction but the wrong size (Singapore −243.49% here vs −240.31% printed), and Thailand, which that report quotes, is dropped from every year here because its Services column is zero (Section C).
- **The reported "% change in ECI" figures are mostly sign-flip artifacts.** Singapore's notebook ECI goes from 0.76 to −1.1 (−243.49%). With the sign fixed, it goes from 0.76 to 1.1 (+43.49%).

## Why this matters for the lab

Every later question (Q02–Q11) needs one trusted ECI implementation, so this run is the foundation. It also tests the most basic link in the chain: **is the number in our reports what we think it is?** Two parts of the answer bear on the reports directly:

1. The sign of the grouped ECI the reports plot and quote is arbitrary from year to year (it depends on which country lands in row 0). Any year-over-year "ECI change", the ECI-vs-GDP figure, and the `[ECI growth quality]` label compare numbers with possibly opposite signs. The greedy optimizer re-runs `compute_eci` on every perturbed matrix, so if the sign flips between candidates it would compare opposite-signed numbers. That hasn't been tested yet (see avenues).
2. We can't currently trace the Spring 2026 report ECIs to the data in the repo. CLAUDE.md rule 5 requires that traceability.

## Method

**Pre-stated checks and pass criteria** (written before running):

| Check | Metric | Pass if |
|---|---|---|
| A. Port vs original notebook code | same country set; max abs ECI difference, per year | identical sets, diff < 1e-9 |
| B. Eigenvector diagnostics | rank of the eigenvalue the notebook picks; sign vs diversity; corr(notebook, oriented) | rank = 2 every year (sign is reported, not pass/fail) |
| C. Published reports | same-country match within 0.0015 (3-dp rounding) against any year × variant | at least the 2019 value of each Spring 2026 country matches |
| D. Maitree's results | max abs diff of `eci_hs4` / `eci_grp` on ISO3; Spearman ρ with log GDPpc (PPP) | diff < 1e-9; ρ equal to 4 dp |

**Baseline:** the notebook as-is (`notebook_eci`). **Library default (`eci_pci` / `eci_from_exports`):** the symmetric form B = D_c^-1/2 M D_p^-1/2, with an SVD so the eigenvalues come out sorted, the 2nd singular vector, and the sign oriented so that ECI correlates positively with diversity (PCI negatively with ubiquity).

**Variants searched in C** (each with and without the High-complexity aggregate, 2019–2023, so 30 in total): the notebook as-is; oriented with strict RCA and the notebook zero filter; oriented with RCA ≥ 1 and no zero filter.

In [1]:
import sys, time, ast, re, json, subprocess, operator
from pathlib import Path
import numpy as np, pandas as pd
from scipy import linalg as LA
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
T0 = time.time()
sys.path.insert(0, '../../lib')
import eci
REPO = eci.REPO
YEARS = [2019, 2020, 2021, 2022, 2023]
pd.set_option('display.width', 160)
print('repo:', REPO)
print('numpy', np.__version__, '| pandas', pd.__version__)

repo: /home/user/eci-research
numpy 2.4.6 | pandas 3.1.1


## Results

### A. Does the library port match the original notebook code exactly?

The original `df_to_matrix`, `compute_mcp`, `compute_mcc`, `compute_eci` and `get_eci_dict` are parsed straight out of `code/current/Spring2026.ipynb` (first definition of each, as the report pipeline uses) and executed unmodified on the same data.

In [2]:
nb = json.load(open(REPO / 'code/current/Spring2026.ipynb'))
WANT = ['df_to_matrix', 'compute_mcp', 'compute_mcc', 'compute_eci', 'get_eci_dict']
src = {}
for cell in nb['cells']:
    if cell['cell_type'] != 'code':
        continue
    s = ''.join(cell['source'])
    try:
        tree = ast.parse(s)
    except SyntaxError:      # cells with ! shell magics
        continue
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name in WANT and node.name not in src:
            src[node.name] = ast.get_source_segment(s, node)
ns = {'np': np, 'LA': LA, 'operator': operator}
for name in WANT:
    exec(src[name], ns)
print('extracted from notebook:', sorted(src))

rows = []
nb_eci = {}
for y in YEARS:
    df = pd.read_csv(REPO / 'data/archive/2024-fall/2019-2023 cleaned' / f'{y}clean.csv')  # the path the notebook reads
    df['Country'] = df['Country'].replace(eci.NOTEBOOK_RENAMES)                           # the notebook's rename cell
    orig = dict(ns['get_eci_dict'](ns['df_to_matrix'](df)))
    port = eci.notebook_eci(eci.load_groups(y))
    nb_eci[y] = port
    common = sorted(set(orig) & set(port.index))
    diff = max(abs(orig[c] - port[c]) for c in common)
    rows.append({'year': y, 'n_original': len(orig), 'n_port': len(port),
                 'same_country_set': set(orig) == set(port.index), 'max_abs_diff': diff})
A = pd.DataFrame(rows)
print(A.to_string(index=False))

extracted from notebook: ['compute_eci', 'compute_mcc', 'compute_mcp', 'df_to_matrix', 'get_eci_dict']


 year  n_original  n_port  same_country_set  max_abs_diff
 2019         121     121              True  2.720046e-15
 2020         121     121              True  5.329071e-15
 2021         106     106              True  6.217249e-15
 2022         121     121              True  3.663736e-15
 2023         123     123              True  3.996803e-15


### B. What does the notebook's eigenvector actually pick, and which way does it point?

In [3]:
rows = []
std_eci = {}
for y in YEARS:
    X = eci.notebook_filter(eci.load_groups(y))
    s, d = eci.notebook_eci(X, apply_filter=False, return_diag=True)
    e, p, dd = eci.eci_pci(eci.notebook_mcp(X).pipe(lambda m: m.loc[:, m.sum() > 0]))
    std_eci[y] = e
    rows.append({'year': y, 'n_countries': d['n_countries'], 'col1_eigen_rank': d['col1_eigen_rank'],
                 'lambda2': round(dd['lambda2'], 4), 'lambda3': round(dd['lambda3'], 4),
                 'eigengap_23': round(dd['eigengap_23'], 4),
                 'notebook_sign_agrees_with_diversity': d['sign_agrees_with_diversity'],
                 'corr(notebook, oriented)': round(np.corrcoef(s, e.reindex(s.index))[0, 1], 6)})
B = pd.DataFrame(rows)
print(B.to_string(index=False))

 year  n_countries  col1_eigen_rank  lambda2  lambda3  eigengap_23  notebook_sign_agrees_with_diversity  corr(notebook, oriented)
 2019          121                2   0.3320   0.2496       0.0823                                 True                       1.0
 2020          121                2   0.3374   0.2399       0.0975                                 True                       1.0
 2021          106                2   0.3385   0.2691       0.0693                                False                      -1.0
 2022          121                2   0.3459   0.2754       0.0705                                False                      -1.0
 2023          123                2   0.3412   0.2160       0.1252                                False                      -1.0


### C. Can we reproduce the ECI numbers printed in finished reports?

Report values are pulled from the PDFs with `pdftotext` and a regex, so nothing is typed in by hand.

In [4]:
SPRING = ['Canada', 'Germany', 'Poland', 'Singapore', 'United Arab Emirates', 'United Kingdom', 'Vietnam']
pat = re.compile(r"ECI (?:increased|decreased) from (-?[0-9.]+) in (\d{4}) to (-?[0-9.]+) in (\d{4})")
rows = []
for c in SPRING:
    txt = subprocess.run(['pdftotext', '-layout', str(REPO / 'reports/2026-spring' / f'{c}.pdf'), '-'],
                         capture_output=True, text=True).stdout
    m = pat.search(' '.join(txt.split()))
    v0, y0, v1, y1 = float(m.group(1)), int(m.group(2)), float(m.group(3)), int(m.group(4))
    rows.append({'country': c, 'report_start': f'{v0:+.3f} ({y0})', 'report_end': f'{v1:+.3f} ({y1})',
                 'port_2019': round(nb_eci[2019].get(c, np.nan), 3), 'port_2023': round(nb_eci[2023].get(c, np.nan), 3),
                 'oriented_2019': round(std_eci[2019].get(c, np.nan), 3), 'oriented_2023': round(std_eci[2023].get(c, np.nan), 3)})
C1 = pd.DataFrame(rows)
print(C1.to_string(index=False))

# Is any report value equal (to 3 dp) to the SAME country's value in ANY year / variant we can build?
variants = {}
for y in YEARS:
    for drop in (False, True):
        X = eci.load_groups(y, drop_aggregate=drop)
        variants[(y, drop, 'notebook')] = eci.notebook_eci(X)
        variants[(y, drop, 'oriented, strict RCA, zero-filter')] = eci.eci_from_exports(eci.notebook_filter(X), strict=True)
        variants[(y, drop, 'oriented, no zero-filter')] = eci.eci_from_exports(X)
n_hits = 0
for _, r in C1.iterrows():
    for val in (float(r.report_start.split()[0]), float(r.report_end.split()[0])):
        for k, s in variants.items():
            if r.country in s.index and abs(s[r.country] - val) < 0.0015:
                n_hits += 1
                print('match:', r.country, val, k)
print(f'variants searched: {len(variants)}; same-country matches to 3 dp: {n_hits}')

             country  report_start    report_end  port_2019  port_2023  oriented_2019  oriented_2023
              Canada +0.858 (2019) -0.456 (2024)     -0.853      0.373         -0.853         -0.373
             Germany -2.834 (2019) +1.309 (2024)      1.988     -1.505          1.988          1.505
              Poland -0.797 (2019) +1.077 (2024)      0.951     -0.635          0.951          0.635
           Singapore -1.276 (2019) +2.417 (2024)      0.764     -1.096          0.764          1.096
United Arab Emirates +0.675 (2019) -0.255 (2024)     -2.079      2.228         -2.079         -2.228
      United Kingdom -3.281 (2019) +1.988 (2024)      0.509      2.618          0.509         -2.618
             Vietnam -1.184 (2019) +1.016 (2024)      0.741     -0.459          0.741          0.459


variants searched: 30; same-country matches to 3 dp: 0


In [5]:
# Fall 2025 Singapore report (2019-2023 data): "% change in ECI" = (last - first) / |first| * 100, as in notebook cell #26
txt = (REPO / 'catalog/text/reports/2025-fall/Singapore/Singapore_2023.md').read_text()
line = [l for l in txt.splitlines() if '% decrease in ECI' in l or '% increase' in l][0]
print('report:', line.strip())
rows = []
for c in ['Singapore', 'China', 'Indonesia', 'Thailand']:
    path = [(y, nb_eci[y][c]) for y in YEARS if c in nb_eci[y].index]
    pct = (path[-1][1] - path[0][1]) / abs(path[0][1]) * 100 if path else np.nan
    pathO = [(y, std_eci[y][c]) for y in YEARS if c in std_eci[y].index]
    pctO = (pathO[-1][1] - pathO[0][1]) / abs(pathO[0][1]) * 100 if pathO else np.nan
    rows.append({'country': c, 'years_present': len(path), 'notebook_path': [round(v, 2) for _, v in path],
                 'notebook_pct_change': round(pct, 2), 'oriented_path': [round(v, 2) for _, v in pathO],
                 'oriented_pct_change': round(pctO, 2)})
print(pd.DataFrame(rows).to_string(index=False))
G19 = eci.load_groups(2019)
print('Thailand 2019 zero columns:', list(G19.columns[G19.loc['Thailand'] <= 0]))
print('countries dropped by the zero filter:', {y: len(eci.load_groups(y)) - len(nb_eci[y]) for y in YEARS})

report: Singapore has seen a 240.31% decrease in ECI. Thailand and China saw their ECI decrease by 617.72% and 184.44% respectively, while Indonesia saw a 181.39% increase.
  country  years_present                     notebook_path  notebook_pct_change                       oriented_path  oriented_pct_change
Singapore              5  [0.76, 0.54, -1.13, -1.52, -1.1]              -243.49       [0.76, 0.54, 1.13, 1.52, 1.1]                43.49
    China              5 [1.39, 1.18, -1.33, -1.04, -0.96]              -169.20      [1.39, 1.18, 1.33, 1.04, 0.96]               -30.80
Indonesia              5  [-0.35, -0.46, 0.43, 0.35, 0.31]               190.64 [-0.35, -0.46, -0.43, -0.35, -0.31]                 9.36
 Thailand              0                                []                  NaN                                  []                  NaN
Thailand 2019 zero columns: ['Services and Utilities']
countries dropped by the zero filter: {2019: 40, 2020: 34, 2021: 31, 2022: 21, 2023: 32

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, (src_, title) in zip(axes, [(nb_eci, 'Notebook as-is (sign set by row 0)'), (std_eci, 'Library: oriented to diversity')]):
    for c in SPRING:
        ax.plot(YEARS, [src_[y].get(c, np.nan) for y in YEARS], marker='o', label=c)
    ax.axhline(0, color='grey', lw=0.6)
    ax.axvspan(2020.5, 2023.5, color='0.92', zorder=-1)
    ax.set_title(title, fontsize=10); ax.set_xticks(YEARS)
axes[0].set_ylabel('grouped ECI (18 groups)')
fig.legend(*axes[0].get_legend_handles_labels(), fontsize=7, loc='lower center', ncol=7, frameon=False)
fig.suptitle('Spring 2026 report countries: shaded years are where the notebook sign is flipped', fontsize=10)
fig.tight_layout(rect=(0, 0.07, 1, 1)); fig.savefig('figures/spring2026_eci_paths.png', dpi=110); plt.show()
print('saved figures/spring2026_eci_paths.png')

saved figures/spring2026_eci_paths.png


### D. Does the library reproduce Maitree's HS4 and grouped ECI and her income correlations?

Same settings as `maitreemody/analyze.py`: HS4 without chapters 98/99, reporters with ≥ $1B exports, RCA ≥ 1, sign oriented to diversity; grouped ECI drops the High-complexity aggregate, maps names to M49 codes, ≥ $1B exports, no zero filter.

In [7]:
mm = pd.read_csv(REPO / 'maitreemody/outputs/country_year_metrics.csv')
mcorr = pd.read_csv(REPO / 'maitreemody/outputs/correlations.csv')
wb = pd.read_csv(REPO / 'maitreemody/outputs/worldbank_gdp_per_capita_2019_2023.csv')
n2c = eci._name_to_m49()
rows, corr_rows, frames = [], [], []
for y in YEARS:
    X = eci.load_hs4(y)
    h = eci.eci_from_exports(X).rename('eci_hs4').to_frame()
    h['iso3'] = [eci.m49_to_iso3(c) for c in h.index]
    G = eci.load_groups(y, drop_aggregate=True, notebook_names=False)
    G.index = [n2c.get(n.strip()) for n in G.index]
    G = G.loc[[c is not None and c not in eci.M49_AGGREGATES for c in G.index]]
    G = G.loc[G.sum(axis=1) >= 1e9]
    g = eci.eci_from_exports(G).rename('eci_grp').to_frame()
    g['iso3'] = [eci.m49_to_iso3(c) for c in g.index]
    d = h.merge(g, on='iso3', how='outer').assign(year=y)
    frames.append(d)
    ref = mm[mm.year == y]
    for col in ['eci_hs4', 'eci_grp']:
        j = d[['iso3', col]].dropna().merge(ref[['iso3', col]].dropna(), on='iso3', suffixes=('_lib', '_maitree'))
        rows.append({'year': y, 'measure': col, 'n_matched': len(j),
                     'max_abs_diff': (j[col + '_lib'] - j[col + '_maitree']).abs().max()})
    w = wb[wb.year == y][['iso3', 'gdppc_ppp']]
    for col, label in [('eci_hs4', 'ECI, HS4 products'), ('eci_grp', 'ECI, 17 groups (sign fixed)')]:
        t = d[['iso3', col]].dropna().merge(w, on='iso3').dropna()
        rho = t[col].corr(np.log(t.gdppc_ppp), method='spearman')
        r = mcorr[(mcorr.year == y) & (mcorr.measure == label)].iloc[0]
        # tie-robust: identical M rows give ECIs equal up to ~1e-15 noise; round before ranking
        rho9 = t[col].round(9).corr(np.log(t.gdppc_ppp), method='spearman')
        tm = ref[[col, 'log_gdppc']].dropna()
        rho9_m = tm[col].round(9).corr(tm.log_gdppc, method='spearman')
        corr_rows.append({'year': y, 'measure': label, 'n_lib': len(t), 'n_maitree': int(r.n),
                          'rho_lib': round(rho, 4), 'rho_maitree_csv': round(r.spearman_rho, 4),
                          'rho_lib_r9': round(rho9, 4), 'rho_maitree_r9': round(rho9_m, 4),
                          'tied_groups_eci': int(t[col].round(9).duplicated().sum())})
D1 = pd.DataFrame(rows); D2 = pd.DataFrame(corr_rows)
print(D1.to_string(index=False)); print(); print(D2.to_string(index=False))

 year measure  n_matched  max_abs_diff
 2019 eci_hs4        131  9.325873e-15
 2019 eci_grp        127  3.996803e-15
 2020 eci_hs4        128  1.054712e-14
 2020 eci_grp        125  9.436896e-15
 2021 eci_hs4        134  1.454392e-14
 2021 eci_grp        118  5.773160e-15
 2022 eci_hs4        117  1.164693e-14
 2022 eci_grp        118  4.440892e-15
 2023 eci_hs4        129  1.032507e-14
 2023 eci_grp        128  8.437695e-15

 year                     measure  n_lib  n_maitree  rho_lib  rho_maitree_csv  rho_lib_r9  rho_maitree_r9  tied_groups_eci
 2019           ECI, HS4 products    131        131   0.7795           0.7795      0.7795          0.7795                0
 2019 ECI, 17 groups (sign fixed)    127        127   0.3730           0.3718      0.3723          0.3723                7
 2020           ECI, HS4 products    128        128   0.8057           0.8057      0.8057          0.8057                0
 2020 ECI, 17 groups (sign fixed)    125        125   0.3995           0.3998 

### E. Library quick reference

In [8]:
X = eci.load_hs4(2019)
e, p, d, Mx = eci.eci_from_exports(X, return_all=True)
print('2019 HS4:', d)
print('top 6 ECI:', [eci.m49_to_iso3(c) for c in e.nlargest(6).index])
print('corr(ECI, diversity) =', round(np.corrcoef(e, Mx.sum(axis=1))[0, 1], 3))
print('functions:', [f for f in dir(eci) if not f.startswith('_') and callable(getattr(eci, f)) and getattr(eci, f).__module__ == 'eci'])
print(f'run time: {time.time() - T0:.1f} s')

2019 HS4: {'lambda1': 0.999999999999998, 'lambda2': 0.26196744955838336, 'lambda3': 0.18203597797024734, 'eigengap_23': 0.07993147158813602, 'n_countries': 133, 'n_products': 1224}
top 6 ECI: ['JPN', 'KOR', 'CHE', 'TWN', 'SGP', 'DEU']
corr(ECI, diversity) = 0.67
functions: ['eci_from_exports', 'eci_pci', 'group_name_to_iso3', 'load_groups', 'load_hs4', 'm49_to_iso3', 'mcp', 'notebook_eci', 'notebook_filter', 'notebook_mcp', 'rca']
run time: 4.4 s


## Caveats

- Check C only shows that the report numbers **don't come from these files with these methods**. It can't say what they did come from. The likeliest explanation is a 2024 data pull and/or differently cleaned CSVs that aren't in the repo (reports say "2019 to 2024"). A near-match to 3 dp elsewhere (e.g. another country's value) is chance, so only same-country matches were counted.
- Section B's "column 1 = 2nd-largest eigenvalue" holds for these five matrices. `scipy.linalg.eig` doesn't guarantee an order, so it could break with other data (e.g. a perturbed matrix inside the greedy loop). The library sorts explicitly.
- Grouped ECI has many exact ties: countries with identical 0/1 rows across the 18 groups (4–9 tied values per year in Maitree's sample). Rank statistics on grouped ECI should round first, or they'll vary at the 3rd decimal between machines.
- 2023 values are reported but are suspect (D1). The UK's oriented grouped ECI is 0.509 in 2019 but −2.618 in 2023 (Section C table), which is worth a look under Q06.
- Nominal export values throughout, and no population filter.

## Helpful to know

- **Use the library:** `sys.path.insert(0, '../../lib'); import eci`. `eci.load_groups(y, drop_aggregate=...)`, `eci.load_hs4(y)` (M49 code × HS4, ≥ $1B exports, no chapters 98/99), `eci.eci_from_exports(X)` (oriented ECI), `eci.eci_pci(M)` (ECI, PCI, eigen diagnostics incl. the λ2−λ3 gap), `eci.notebook_eci(X)` (exact notebook reproduction), `eci.m49_to_iso3`, `eci.group_name_to_iso3` (use on raw CSV names, *before* the notebook renames).
- The notebook's zero filter (`df_to_matrix`) drops 40 / 34 / 31 / 21 / 32 countries in 2019–2023. **Thailand is dropped in every year** because its *Services and Utilities* value is 0, so the current notebook can't produce a Thailand report from these files.
- The notebook RCA is strict (`>`), while Maitree/Atlas use `≥`. On grouped data the difference only matters at exact equality, but keep it explicit (`strict=`).
- The notebook's percentage "ECI change" (cell #26) divides by |first value|. With values near zero, or with a sign flip in between, it produces numbers like −617.72% that mean nothing.
- `pip` in this container installs into Python 3.13, but `python3`/Jupyter run 3.11. Use `python3 -m pip install ...`. A plain `pip install -r requirements.txt` also hit a hash-mismatch error that left numpy uninstalled.

## New research avenues (each about one night)

1. **Greedy sign stability:** without running the full optimizer, count how often `compute_eci`'s sign flips between the base matrix and the single-industry perturbations that `two_step_greedy` evaluates for each Spring 2026 country (one budget share, one year). A flip there would make the optimizer chase the wrong direction.
2. **Re-derive the Spring 2026 report claims with the oriented ECI:** recompute the ECI paths, the "% change" and `[ECI growth quality]` labels for the 7 countries and their peers on 2019–2022, and list which report sentences change direction.
3. **Zero-filter alternative:** compare the notebook's "drop any zero" rule against keeping zeros (RCA handles them) on country coverage and rank stability. This feeds Q03.
4. **Tie structure of grouped ECI:** how many distinct ECI values exist per year at 18 groups, i.e. how many countries are indistinguishable. A one-cell result that quantifies the granularity problem ahead of Q04.

## Data issues / data needed

- **Spring 2026 report ECIs can't be traced to repo data** (Section C). Request: commit (or point to) the exact cleaned CSVs, including any 2024 file, and the `<Country>_eci_19to24_dict.txt` / `.xlsx` outputs used for the 7 Spring 2026 reports and the Fall 2025 Singapore and Thailand reports. See `willbetts/nightly/data-requests/2026-11-11-report-eci-provenance.md`.
- **Thailand has 0 in *Services and Utilities* in all five cleaned files**, so the notebook drops it. The Fall 2025 reports nevertheless quote Thailand's ECI. Same data request.
- **Notebook sign bug** (`compute_eci`, cells #14/#58): this is a code issue, not data. Suggest adding it to KNOWN-ISSUES as N10. The fix is to orient by correlation with diversity, as `eci.eci_pci` does. Not changed here, because this run may only add files under `willbetts/nightly/`.